# EEG datasets

> Where open EEG data lives, and how much of it there is

This page catalogs open EEG data: the places it is published, and the datasets themselves with their rough size. It is a map for finding data, not a benchmark. Every number comes from the dataset's own page or paper, and a blank means the source doesn't report it.

Benchmark libraries cover one paradigm family each: 200 of MOABB's 206 datasets, for example, are BCI paradigms (motor imagery, ERP, SSVEP and c-VEP), and it loads none of the large clinical or sleep corpora. Labs also tend to publish to one repository each, so no single source sees everything. The tables below span EEG repositories, general-purpose repositories, clinical corpora and lab websites.

The catalog is two CSV files in the repo, [`catalog/sources.csv`](https://github.com/bkowshik/temple-of-eeg-decoding/blob/main/catalog/sources.csv) and [`catalog/datasets.csv`](https://github.com/bkowshik/temple-of-eeg-decoding/blob/main/catalog/datasets.csv), so you can load, filter or correct it. Each row there also records the license, the reference, the page its numbers came from, and notes on how they were counted, including which numbers are approximate. Numbers were gathered on 3 November 2026.

**Access:**

- **open**: download directly, no account
- **registration**: free account or login
- **DUA**: signed data use agreement or credentialed access
- **request**: apply to the lab or authors

**Trials** counts trials across all subjects and sessions; **hours** is total recording time. Dataset tables are sorted by number of subjects, and each name links to the dataset's landing page.

In [ ]:
#| hide
import csv
from collections import Counter, defaultdict
from pathlib import Path
from IPython.display import Markdown

In [ ]:
#| hide
catalog = Path('../catalog')
sources = list(csv.DictReader(open(catalog/'sources.csv')))
datasets = list(csv.DictReader(open(catalog/'datasets.csv')))

In [ ]:
#| hide
def num(s):
    "Format a CSV number for display; blank stays blank"
    if not s: return ''
    x = float(s)
    return f'{x:,.0f}' if x >= 10 or x == int(x) else f'{x:,.1f}'

def esc(s): return s.replace('|', '\\|')

def note(s):
    "Escape a CSV note, spelling out the `approx:` marker"
    return '*Approximate:* ' + esc(s[8:]) if s.startswith('approx: ') else esc(s)

def link(name, url): return f'[{esc(name)}]({url})' if url else esc(name)

def table(rows, cols):
    "Markdown table of `rows`; `cols` holds (header, row -> str, is_numeric) triples"
    lines = ['| ' + ' | '.join(h for h,_,_ in cols) + ' |',
             '|' + ''.join('---:|' if n else '---|' for _,_,n in cols)]
    lines += ['| ' + ' | '.join(f(r) for _,f,_ in cols) + ' |' for r in rows]
    return Markdown('\n'.join(lines))

def by_size(r): return -float(r['subjects'] or 0), r['name'].lower()

## At a glance

How many datasets the catalog holds for each paradigm, and how many of them you can download without an account.

In [ ]:
#| echo: false
order = [p for ps in {'Brain-computer interfaces': ['Motor imagery', 'Motor execution', 'ERP / P300', 'SSVEP', 'c-VEP'], 'Clinical, sleep and resting state': ['Epilepsy & clinical', 'Sleep', 'Resting state', 'Large corpus'], 'Cognition, emotion and perception': ['Affective', 'Visual perception', 'Auditory & speech', 'Language', 'Cognitive']}.values() for p in ps]
counts = Counter(r['paradigm'] for r in datasets)
open_counts = Counter(r['paradigm'] for r in datasets if r['access'] == 'open')
biggest = {p: max((r for r in datasets if r['paradigm'] == p), key=lambda r: float(r['subjects'] or 0)) for p in counts}
rows = [dict(paradigm=p, n=str(counts[p]), open=str(open_counts[p]), big=biggest[p]) for p in order if counts[p]]
rows.append(dict(paradigm='**All**', n=f'**{len(datasets)}**', open=f"**{sum(open_counts.values())}**", big=None))
table(rows, [('Paradigm', lambda r: r['paradigm'], False),
             ('Datasets', lambda r: r['n'], True),
             ('Open', lambda r: r['open'], True),
             ('Most subjects', lambda r: f"{link(r['big']['name'], r['big']['url'])} ({num(r['big']['subjects'])})" if r['big'] else '', False)])

| Paradigm | Datasets | Open | Most subjects |
|---|---:|---:|---|
| Motor imagery | 28 | 28 | [PhysioNet EEG Motor Movement/Imagery (EEGBCI)](https://physionet.org/content/eegmmidb/1.0.0/) (109) |
| Motor execution | 4 | 4 | [Schwarz 2020 reach-and-grasp](https://bnci-horizon-2020.eu/database/data-sets) (45) |
| ERP / P300 | 18 | 17 | [bigP3BCI (Mainsah 2025)](https://physionet.org/content/bigp3bci/1.0.0/) (326) |
| SSVEP | 9 | 9 | [Wearable SSVEP dataset (Zhu 2021)](https://doi.org/10.6084/m9.figshare.13560281) (102) |
| c-VEP | 5 | 5 | [Zheng 2024 narrow-band c-VEP](https://doi.org/10.6084/m9.figshare.24864243) (100) |
| Epilepsy & clinical | 26 | 15 | [NMT Scalp EEG Dataset](https://dll.seecs.nust.edu.pk/downloads/) (2,417) |
| Sleep | 17 | 8 | [Human Sleep Project (HSP)](https://bdsp.io/content/hsp/) (90,166) |
| Resting state | 5 | 4 | [Dortmund Vital Study resting EEG (ds005385)](https://openneuro.org/datasets/ds005385) (608) |
| Large corpus | 3 | 1 | [Harvard Electroencephalography Database (HEEDB)](https://bdsp.io/content/harvard-eeg-db/) (109,178) |
| Affective | 14 | 1 | [FACED](https://doi.org/10.7303/syn50614194) (123) |
| Visual perception | 10 | 9 | [THINGS-EEG1](https://openneuro.org/datasets/ds003825) (50) |
| Auditory & speech | 14 | 14 | [SparrKULee](https://doi.org/10.48804/K3VSND) (85) |
| Language | 6 | 6 | [Alice EEG (Brennan)](https://deepblue.lib.umich.edu/data/concern/data_sets/bg257f92t) (49) |
| Cognitive | 7 | 5 | [STEW](https://ieee-dataport.org/open-access/stew-simultaneous-task-eeg-workload-dataset) (48) |
| **All** | **166** | **126** |  |

Where the cataloged datasets are hosted. No single repository holds most of them, which is why a catalog built from any one source misses a lot.

In [ ]:
#| echo: false
hosts = defaultdict(list)
for r in datasets: hosts[r['source']].append(r)
rows = sorted(hosts.items(), key=lambda kv: (-len(kv[1]), kv[0].lower()))
singles = [h for h,rs in rows if len(rs) == 1]
rows = [(h, str(len(rs)), ', '.join(p for p,_ in Counter(r['paradigm'] for r in rs).most_common())) for h,rs in rows if len(rs) > 1]
rows.append((f'{len(singles)} other hosts', str(len(singles)), 'One dataset each: ' + ', '.join(singles)))
table(rows, [('Host', lambda r: esc(r[0]), False),
             ('Datasets', lambda r: r[1], True),
             ('Paradigms', lambda r: esc(r[2]), False)])

| Host | Datasets | Paradigms |
|---|---:|---|
| Lab website | 31 | Affective, Motor imagery, Epilepsy & clinical, ERP / P300, SSVEP, Auditory & speech, Sleep, Visual perception, Cognitive |
| OpenNeuro | 23 | Epilepsy & clinical, Auditory & speech, Resting state, Visual perception, Language, Motor imagery, Large corpus, Sleep, Affective |
| Zenodo | 21 | ERP / P300, Auditory & speech, Motor imagery, Sleep, Affective, SSVEP, c-VEP, Epilepsy & clinical, Cognitive |
| Figshare | 14 | Motor imagery, SSVEP, Motor execution, ERP / P300, c-VEP, Epilepsy & clinical, Cognitive |
| BNCI Horizon 2020 | 10 | Motor imagery, ERP / P300, Motor execution |
| PhysioNet | 10 | Sleep, Epilepsy & clinical, Motor imagery, ERP / P300, Cognitive |
| NSRR | 7 | Sleep |
| OSF | 7 | Language, ERP / P300, Visual perception, Auditory & speech, Cognitive |
| TUH EEG | 7 | Epilepsy & clinical, Large corpus |
| GigaDB | 4 | Motor imagery, ERP / P300, SSVEP |
| BDSP | 2 | Large corpus, Sleep |
| Harvard Dataverse | 2 | Motor imagery |
| Hugging Face | 2 | Visual perception |
| IEEE DataPort | 2 | Epilepsy & clinical, Cognitive |
| Kaggle | 2 | ERP / P300, Epilepsy & clinical |
| Radboud Data Repository | 2 | c-VEP |
| Stanford Digital Repository | 2 | Visual perception, Auditory & speech |
| Synapse | 2 | Resting state, Affective |
| 16 other hosts | 16 | One dataset each: Borealis, DataverseNL, Deep Blue Data, DepositOnce (TU Berlin), FreiDok, G-Node GIN, GitHub, Google Drive, INDI (NITRC), KiltHub (Figshare), KU Leuven RDR, NEMAR, Recherche Data Gouv, RepOD, Tsinghua Cloud, Universidad de Valladolid repository |

The split holds even inside one benchmark. MOABB's own table of its 206 datasets puts 54 on Zenodo, 19 of them from French labs such as GIPSA-lab and Inria. It puts 21 on PhysioNet, 20 of which are studies from a single P300 collection, bigP3BCI. All 19 on BNCI Horizon 2020 come from European labs, 8 of them Austrian. Figshare holds 17, OSF 13, and Harvard Dataverse 10, 7 of them from Chinese labs. All 4 on GigaDB come from Korean labs. Only 3 are on OpenNeuro, and 31 name no repository at all.

## Where EEG data lives

Three kinds of places publish EEG data. Counts are as each source reported them on 3 November 2026.

In [ ]:
#| hide
def group(g): return sorted((r for r in sources if r['group'] == g), key=lambda r: (-float(r['eeg_datasets'] or 0), r['name'].lower()))

### EEG-focused repositories and corpora

Repositories and corpora built for neural or clinical data. Counts are of EEG datasets, taken from each source's API or catalog.

In [ ]:
#| echo: false
#| column: page
table(group('EEG-focused'), [('Source', lambda r: link(r['name'], r['url']), False),
                             ('EEG datasets', lambda r: num(r['eeg_datasets']), True),
                             ('Subjects', lambda r: num(r['subjects']), True),
                             ('Hours', lambda r: num(r['hours']), True),
                             ('Access', lambda r: r['access'], False),
                             ('Python access', lambda r: esc(r['programmatic_access']), False),
                             ('Notes', lambda r: note(r['notes']), False)])

| Source | EEG datasets | Subjects | Hours | Access | Python access | Notes |
|---|---:|---:|---:|---|---|---|
| [NEMAR](https://nemar.org) | 652 | 35,750 | 105,150 | open | REST API (api.nemar.org/datasets); HTTPS file tree (data.nemar.org); Zarr; EEGDash | *Approximate:* hours sum total_recording_duration for the 619 of 652 EEG datasets that report it. Mirrors 434 OpenNeuro EEG datasets ('on' IDs) and now hosts 218 of its own ('nm' IDs, 206 from one uploader), mostly BIDS conversions of BCI, PhysioNet and sleep datasets from other repositories. |
| [OpenNeuro](https://openneuro.org) | 468 | 27,369 |  | open | GraphQL API (openneuro.org/crn/graphql); openneuro-py; DataLad; S3 | *Approximate:* 468 of 1,935 readable public datasets list eeg among snapshot modalities, subjects summed per dataset (the API's modality filter alone missed HBN Release 7); no hours field, but NEMAR's mirror of 434 of them sums about 77,700 h. BIDS-only, so cognitive and clinical research labs dominate: only 3 of MOABB's 206 BCI datasets live here. |
| [Brain Data Science Platform (BDSP)](https://bdsp.io) | 50 |  |  | mixed | web download after credentialing; per-project tools on GitHub (bdsp-core) | *Approximate:* 50 of 72 listed projects carry the 'eeg' topic, many of them paper data-and-code packages; 43 databases are credentialed or restricted, 5 open. Run by Stanford's Clinical Data Animation Center with Harvard/MGH/BIDMC: Harvard EEG Database, Human Sleep Project v3.0 (119,234 PSG recordings, 90,166 patients), I-CARE, SPaRCNet. |
| [National Sleep Research Resource (NSRR)](https://sleepdata.org) | 42 | 50,951 |  | DUA | NSRR Ruby gem; nsrr R package; Luna | *Approximate:* 42 of 55 listed datasets are tagged Polysomnography, subjects summed from their cards; some are animal studies and some (e.g., HCHS, recorded on an ARES Unicorder) are home sleep tests, so not all have full EEG. Mostly US sleep cohorts and trials (SHHS, MESA, MrOS, CHAT, NCHSDB); the site carries a banner saying it is 'under review for potential modification'. |
| [Kaggle competitions](https://www.kaggle.com/competitions?search=EEG) | 37 |  |  | registration | Kaggle API/CLI | 37 competitions match 'EEG'; the research-tier EEG ones are seizure-detection and seizure-prediction (2014), grasp-and-lift-eeg-detection and inria-bci-challenge (2015), melbourne-university-seizure-prediction (2016) and hms-harmful-brain-activity-classification (2024). Data needs a Kaggle account and accepting each competition's rules. |
| [BNCI Horizon 2020 database](http://bnci-horizon-2020.eu/database/data-sets) | 29 | 344 |  | open | direct HTTP download; MOABB BNCI loaders | 29 of 32 entries carry EEG (others fNIRS or ECoG); the live site timed out today, so counts come from the 2026-10-12 Wayback snapshot. Graz University of Technology contributed 11 of the EEG sets and TU Berlin 8, almost all BCI paradigms. |
| [PhysioNet](https://physionet.org) | 29 |  |  | mixed | REST project list (physionet.org/api/v1/project/published/); wfdb-python; wget | *Approximate:* 28 databases plus the 2018 Challenge whose metadata mention EEG or polysomnography, after removing 7 false positives; 24 are open, the rest restricted or credentialed. Sleep (Sleep-EDF, CAP, HMC), epilepsy (CHB-MIT, Siena), cardiac arrest (I-CARE) and US BCI data (EEGMMIDB, bigP3BCI); all 21 MOABB datasets sourced here are from US labs. |
| [BBCI BCI Competitions II-IV](https://www.bbci.de/competition/) | 16 |  |  | open | direct download after click-through terms; MOABB and braindecode loaders for some | 16 EEG data sets across Competitions II (6), III (7 of 8, one ECoG) and IV (3 of 5, one MEG, one ECoG), from Berlin, Graz, Wadsworth and Tuebingen groups. Still the default BCI benchmarks (IV-2a is BNCI 001-2014); Competition I is hosted at Columbia. |
| [PREDiCT (Patient Repository for EEG Data + Computational Tools)](http://predict.cs.unm.edu) | 14 |  |  | open | direct download links | *Approximate:* the 'Number of Patients' column sums to 570 across 14 task and rest datasets (Parkinson's, mTBI, depression, OCD, schizophrenia) from the University of New Mexico. PDDL public-domain licence; 64-channel BrainVision or Neuroscan recordings. |
| [Tsinghua University BCI Lab downloads](https://bci.med.tsinghua.edu.cn/download.html) | 12 |  |  | open | direct download | 12 datasets, mostly SSVEP (Benchmark, BETA, eldBETA, wearable, 160-target) plus RSVP and collaborative BCI. Copies reappear via Zenodo/Figshare in MOABB (Wang2016, Liu2022EldBETA) and TorchEEG (TSUBenchmark). |
| [Healthy Brain Network (Child Mind Institute)](https://fcon_1000.projects.nitrc.org/indi/cmi_healthy_brain_network/) | 11 | 4,140 |  | mixed | NITRC/AWS for raw files; OpenNeuro ds005505-ds005516; EEGDash | *Approximate:* subjects sum the non-BIDS EEG column of the HBN download table for releases 1.1-11 (the 11 BIDS releases on OpenNeuro hold 3,155). Child Mind Institute's transdiagnostic paediatric cohort; raw downloads need a NITRC login while the BIDS copies are open. |
| [SJTU BCMI SEED datasets](https://bcmi.sjtu.edu.cn/home/seed/) | 11 |  |  | request | none (application form); TorchEEG loaders after download | 11 SEED-family datasets (emotion, vigilance, cross-cultural, sleep deprivation, video) from Shanghai Jiao Tong University's BCMI lab. Access via an application form; TorchEEG ships loaders for SEED, SEED-IV and SEED-V. |
| [TUH EEG Corpus (Temple University Hospital)](https://isip.piconepress.com/projects/nedc/html/tuh_eeg/) | 7 | 14,987 |  | DUA | rsync over SSH after approval; braindecode TUH/TUHAbnormal/TUHEvents | Official page lists TUEG v2.0.2 (26,846 recordings, 2002-2017) plus 6 labelled subsets (TUAB, TUAR, TUEP, TUEV, TUSZ, TUSL); the patient count is for TUEG v2.0.0 (14,987 patients, 26,846 sessions, 69,652 recordings) from arXiv 2401.10283. Single-hospital clinical archive; access needs a signed form emailed to NEDC, then SSH-key rsync. |
| [European Epilepsy Database (EPILEPSIAE)](https://www.epilepsy-database.eu) | 1 |  | 45,000 | request | none (licensed delivery) | *Approximate:* hours are a lower bound ('> 45,000 hours'); 275 annotated patient datasets from over 250 patients (200+ scalp, 60 intracranial) with over 2,500 seizures. Paid licences only (e.g., 2,500 EUR for 25 surface datasets), now sold via Campus Technologies Freiburg. |
| [Harvard Electroencephalography Database](https://bdsp.io/content/harvard-eeg-db/4.1/) | 1 | 109,178 |  | DUA | BDSP download after credentialing; Harvard-EEG-Database-Tools on GitHub | v4.1 (published 2025-02-10): 284,343 clinical EEG studies from 109,178 patients at four sites, with linked EHR. Requires BDSP credentialed status and a signed project DUA; no hours figure given. |

### General-purpose repositories

Repositories that hold any kind of data and don't tag EEG reliably. The count is search hits for "EEG", which overstates how many EEG datasets they hold: Zenodo's hits even include Germany's renewable-energy law, which is also abbreviated EEG. Much lab-published BCI data lives here all the same.

In [ ]:
#| echo: false
#| column: page
table(group('General-purpose'), [('Source', lambda r: link(r['name'], r['url']), False),
                                 ('Search hits', lambda r: num(r['eeg_datasets']), True),
                                 ('Access', lambda r: r['access'], False),
                                 ('Notes', lambda r: esc(r['notes'].removeprefix('approx: ')), False)])

| Source | Search hits | Access | Notes |
|---|---:|---|---|
| [Figshare](https://figshare.com) | 7,562 | open | items of type dataset returned by the API search (search_for=EEG, item_type=3), which includes journal supplementary files. Hosts 17 MOABB datasets, largely Chinese groups (e.g., Tsinghua eldBETA) plus Greek MAMEM and Mexican BCI data. |
| [OSF (Open Science Framework)](https://osf.io) | 2,689 | open | public OSF projects matching 'EEG' in the SHARE index (cardSearchText=EEG, resourceType=Project; 3,349 with components), many holding code or preregistrations rather than data. Favoured by psychology/ERP labs (ERP CORE) and Korea University's mobile and 2020-competition BCI sets in MOABB. |
| [Kaggle datasets](https://www.kaggle.com/datasets?search=EEG) | 1,558 | registration | unique results of 'kaggle datasets list -s EEG' until pages stopped returning items. Mostly user re-uploads of public datasets (e.g., an OpenNeuro ds002778 copy, SAM40), so check provenance and licence. |
| [Zenodo](https://zenodo.org) | 1,531 | mixed | hits for q=EEG with type=dataset; the query also catches Germany's renewable-energy law (EEG), and 'electroencephalography OR electroencephalogram' gives 421. Largest home of MOABB BCI data (54 of 206), led by French labs such as GIPSA-lab Grenoble (Brain Invaders) and Inria. |
| [Hugging Face Hub](https://huggingface.co/datasets?search=eeg) | 1,001 | mixed | datasets whose name matches 'eeg', of which 738 are EEGDash mirrors of NEMAR/OpenNeuro data; a further 13 datasets sit under the braindecode org. The remaining ~260 are user uploads of mixed provenance and licence. |
| [Harvard Dataverse](https://dataverse.harvard.edu) | 535 | mixed | datasets returned by the Search API for q=EEG, type=dataset. 10 MOABB BCI datasets live here, 7 from Chinese groups (Tianjin University, CAS, World Robot Contest) and 2 from Japan. |
| [Mendeley Data](https://data.mendeley.com) | 413 | open | results attributed to Mendeley Data for query=EEG on its search page (419 across all indexed sources). Elsevier's repository; its 5 MOABB datasets come from Mexico, Ecuador, Indonesia and Turkey. |
| [NIMH Data Archive (NDA)](https://nda.nih.gov) | 278 | DUA | collections matching 'EEG' in NDA's search API (80 match 'electroencephalography'). EEG from NIH-funded mental-health studies (autism, psychosis, depression, adolescent sleep); access needs an institutionally signed Data Use Certification. |
| [IEEE DataPort](https://ieee-dataport.org) | 213 | mixed | results for site search 'EEG': 181 standard datasets (subscription), 24 open access, 8 competitions. Mostly single-lab engineering datasets, often from consumer headsets (e.g., the Emotiv-based STEW workload set). |
| [Dryad](https://datadryad.org) | 157 | open | hits for q=EEG in the Dryad search API (60 for 'electroencephalography'). Journal-linked CC0 deposits, human and animal; few large curated EEG collections. |
| [G-Node GIN](https://gin.g-node.org) | 45 | open | public repositories matching 'EEG' from the GIN API, including forks and DOI copies. Small German/Swiss footprint (MPIB Berlin, USZ Zurich scalp EEG + iEEG, mouse hdEEG). |
| [EBRAINS Knowledge Graph](https://search.kg.ebrains.eu) | 42 | mixed | datasets matching 'EEG' (21 tagged electroencephalography; 13 controlled access, 12 free, 9 embargoed, 8 restricted). European Human Brain Project labs, e.g., TMS-EEG perturbational complexity and sEEG/hd-EEG stimulation studies. |
| [Synapse (Sage Bionetworks)](https://www.synapse.org) | 25 | mixed | public projects matching 'EEG' in the Synapse search API, several unrelated (e.g., mPower). Hosts TD-BRAIN (Brainclinics), BrainLat, HarMNqEEG multinational cross-spectra and FACED; access terms vary by project. |
| [DANDI Archive](https://dandiarchive.org) | 23 | open | dandisets matching 'EEG'; mostly animal and intracranial (iEEG/sEEG, single-unit) recordings. DANDI is the BRAIN Initiative's NWB archive, with little human scalp EEG. |
| [GigaDB](https://gigadb.org) | 8 | open | 8 EEG datasets among the 11 hits for 'EEG' in the GigaDB search API. Data notes for GigaScience; all 4 MOABB datasets here are Korean (Korea University OpenBMI Lee2019, GIST Cho2017). |
| [UCI Machine Learning Repository](https://archive.ics.uci.edu) | 4 | open | 4 datasets match 'EEG' in its API (EEG Database, EEG Eye State, EEG SSVEP signals, BEED epilepsy). Small, old ML-benchmark staples. |

### Aggregators and loader libraries

Catalogs and Python packages that index or load datasets hosted elsewhere, so their counts overlap with the repositories above.

In [ ]:
#| echo: false
#| column: page
table(group('Aggregators & loaders'), [('Source', lambda r: link(r['name'], r['url']), False),
                                       ('Kind', lambda r: r['kind'], False),
                                       ('Datasets', lambda r: num(r['eeg_datasets']), True),
                                       ('Subjects', lambda r: num(r['subjects']), True),
                                       ('Hours', lambda r: num(r['hours']), True),
                                       ('Notes', lambda r: note(r['notes']), False)])

| Source | Kind | Datasets | Subjects | Hours | Notes |
|---|---|---:|---:|---:|---|
| [EEGDash](https://eegdash.org) | aggregator | 697 | 41,083 | 120,657 | *Approximate:* chart-data API totals (691 of 697 with duration), including 11 HBN 'mini' subsets and 11 BDF re-releases of HBN; without them 675 datasets and 37,708 subjects. Indexes NEMAR (653) and OpenNeuro (22) so it inherits their lab mix; the site headline (955 datasets, 137,716 h, all modalities) is slightly ahead of the API (952). |
| [MOABB](https://moabb.neurotechx.com) | loader library | 206 | 4,500 |  | *Approximate:* subjects summed across the 206 dataset classes in the docs table (develop; last release v1.7.2), and some classes share subjects. Its Repository column: Zenodo 54, PhysioNet 21, BNCI 19, Figshare 17, OSF 13, Harvard Dataverse 10, GitHub 7, Mendeley 5, GigaDB 4, OpenNeuro 3, blank 31. |
| [meagmohit/EEG-Datasets (curated list)](https://github.com/meagmohit/EEG-Datasets) | aggregator | 67 |  |  | Curated GitHub list (3,143 stars, last commit 2025-10-17) with 67 numbered entries in 11 paradigm sections plus an unfiltered link dump. A common discovery entry point that mostly points back to the repositories above. |
| [TorchEEG](https://torcheeg.readthedocs.io) | loader library | 21 |  |  | 21 dataset loaders in v1.1.3 (latest, released 2024-12-23), weighted to affective EEG (SEED family, DEAP, DREAMER, AMIGOS, MAHNOB, FACED, MPED) plus sleep (HMC, ISRUC, Sleep-EDFx), BCI and auditory attention (KUL, DTU). Most loaders expect the raw files to be obtained first, several under request-only licences. |
| [braindecode](https://braindecode.org) | loader library | 11 |  |  | v2.0.0 (released 2026-11-02) ships 11 named dataset classes (TUH, TUHAbnormal, TUHEvents, SleepPhysionet, SleepPhysionetChallenge2018, CHBMIT, SIENA, NMT, HGD, BNCI2014_001, BCICompetitionIVDataset4) plus MOABBDataset, BIDSDataset and Hugging Face Hub wrappers. Its clinical and sleep coverage complements MOABB's BCI focus; TUH classes need TUH access. |
| [mne.datasets](https://mne.tools/stable/documentation/datasets.html) | loader library | 9 |  |  | MNE 1.13.2's overview lists 9 datasets with scalp EEG (sample, SPM faces, EEGBCI, mTRF, Kiloword, Sleep PSG, ERP CORE, SSVEP, EyeLink EEG-eyetracking); LIMO is also EEG but not labelled so. Most other fetchers are MEG, OPM, sEEG/ECoG or phantoms; teaching-sized, not benchmarks. |

In [ ]:
#| hide
def paradigm_table(p):
    rows = sorted((r for r in datasets if r['paradigm'] == p), key=by_size)
    return table(rows, [('Dataset', lambda r: link(r['name'], r['url']), False),
                        ('Host', lambda r: esc(r['source']), False),
                        ('Subjects', lambda r: num(r['subjects']), True),
                        ('Trials', lambda r: num(r['trials']), True),
                        ('Hours', lambda r: num(r['hours']), True),
                        ('Channels', lambda r: num(r['channels']), True),
                        ('Hz', lambda r: num(r['sfreq_hz']), True),
                        ('Access', lambda r: r['access'], False),
                        ('Year', lambda r: r['year'], False)])

## Brain-computer interfaces

### Motor imagery

In [ ]:
#| echo: false
#| column: page
paradigm_table('Motor imagery')

| Dataset | Host | Subjects | Trials | Hours | Channels | Hz | Access | Year |
|---|---|---:|---:|---:|---:|---:|---|---|
| [PhysioNet EEG Motor Movement/Imagery (EEGBCI)](https://physionet.org/content/eegmmidb/1.0.0/) | PhysioNet | 109 | 9,838 |  | 64 | 160 | open | 2009 |
| [Dreyer 2023 large MI database](https://doi.org/10.5281/zenodo.8089820) | Zenodo | 87 | 20,800 |  | 27 | 512 | open | 2023 |
| [Stieger 2021 continuous SMR-BCI](https://figshare.com/articles/dataset/Human_EEG_Dataset_for_Brain-Computer_Interface_and_Meditation/13123148) | Figshare | 62 | 269,099 | 600 | 62 | 1,000 | open | 2021 |
| [WBCIC-SHU multi-day MI (Yang 2025)](https://plus.figshare.com/articles/dataset/Brain_Computer_Interface_Motor_Imagery-EEG_Dataset/22671172) | Figshare | 62 | 40,500 |  | 59 | 1,000 | open | 2025 |
| [Leeuwis 2021 MI with psychological factors](https://doi.org/10.34894/Z7ZVOD) | DataverseNL | 55 | 8,800 |  | 16 | 250 | open | 2021 |
| [OpenBMI (Lee 2019) MI](https://doi.org/10.5524/100542) | GigaDB | 54 | 21,600 |  | 62 | 1,000 | open | 2019 |
| [Cho 2017 GIST MI](http://dx.doi.org/10.5524/100295) | GigaDB | 52 | 10,520 |  | 64 | 512 | open | 2017 |
| [Liu 2024 acute stroke MI](https://figshare.com/articles/dataset/EEG_datasets_of_stroke_patients/21679035) | Figshare | 50 | 2,000 |  | 29 | 500 | open | 2024 |
| [Wang 2026 sensory-guided MI training](https://doi.org/10.1184/R1/32293995.v1) | KiltHub (Figshare) | 39 |  |  | 62 | 1,000 | open | 2026 |
| [Iwama 2023 high-density SMR-BCI](https://openneuro.org/datasets/ds004444) | OpenNeuro | 30 | 19,200 |  | 128 | 1,000 | open | 2023 |
| [Zuo 2025 lower-limb MI (knee pain)](https://figshare.com/articles/dataset/28740260) | Figshare | 30 | 15,000 |  | 30 | 500 | open | 2025 |
| [Shin 2017 hybrid EEG-NIRS MI](http://doc.ml.tu-berlin.de/hBCI) | Lab website | 29 | 1,740 |  | 30 | 200 | open | 2017 |
| [Jeong 2020 11 upper-limb tasks](https://zenodo.org/records/19021436) | Zenodo | 25 | 41,250 |  | 60 | 2,500 | open | 2020 |
| [Ma 2020 same-limb joints MI](https://doi.org/10.7910/DVN/RBN3XG) | Harvard Dataverse | 25 | 15,000 |  | 62 | 1,000 | open | 2020 |
| [SHU cross-session MI (Ma 2022)](https://nemar.org/dataexplorer/detail?dataset_id=nm000288) | NEMAR | 25 | 11,988 |  | 32 | 250 | open | 2022 |
| [Zhou 2021 7-day MI](https://zenodo.org/records/18988317) | Zenodo | 20 | 33,600 |  | 26 | 500 | open | 2021 |
| [NETBCI longitudinal MI (Corsi 2026)](https://doi.org/10.57745/RBJRC7) | Recherche Data Gouv | 19 | 14,431 |  | 74 | 250 | open | 2026 |
| [Steyrl 2016 right hand vs feet MI](https://bnci-horizon-2020.eu/database/data-sets) | BNCI Horizon 2020 | 14 | 2,240 |  | 15 | 512 | open | 2016 |
| [Kaya 2018 large MI dataset](https://doi.org/10.6084/m9.figshare.c.3917698) | Figshare | 13 | 60,000 | 60 | 19 | 200 | open | 2018 |
| [Faller 2012 autocalibration MI](https://bnci-horizon-2020.eu/database/data-sets) | BNCI Horizon 2020 | 12 | 5,600 |  | 13 | 512 | open | 2012 |
| [Yi 2014 simple and compound limb MI](http://dx.doi.org/10.7910/DVN/27306) | Harvard Dataverse | 10 | 5,540 |  | 60 | 200 | open | 2014 |
| [BCI Competition IV 2a (Graz)](https://bnci-horizon-2020.eu/database/data-sets) | BNCI Horizon 2020 | 9 | 5,184 |  | 22 | 250 | open | 2012 |
| [BCI Competition IV 2b (Graz)](https://bnci-horizon-2020.eu/database/data-sets) | BNCI Horizon 2020 | 9 | 6,520 |  | 3 | 250 | open | 2007 |
| [Scherer 2015 mental tasks (end users)](https://bnci-horizon-2020.eu/database/data-sets) | BNCI Horizon 2020 | 9 | 3,600 |  | 30 | 256 | open | 2015 |
| [BCI Competition IV 1 (Berlin)](https://www.bbci.de/competition/iv/) | Lab website | 7 |  |  | 59 | 1,000 | open | 2012 |
| [BCI Competition III IVa (Berlin)](https://www.bbci.de/competition/iii/) | Lab website | 5 | 1,400 |  | 118 | 1,000 | open | 2004 |
| [BCI Competition III IIIa (Graz)](https://www.bbci.de/competition/iii/) | Lab website | 3 |  |  | 60 | 250 | open | 2006 |
| [BCI Competition III V (IDIAP)](https://www.bbci.de/competition/iii/) | Lab website | 3 |  | 0.8 | 32 | 512 | open | 2006 |

### Motor execution

In [ ]:
#| echo: false
#| column: page
paradigm_table('Motor execution')

| Dataset | Host | Subjects | Trials | Hours | Channels | Hz | Access | Year |
|---|---|---:|---:|---:|---:|---:|---|---|
| [Schwarz 2020 reach-and-grasp](https://bnci-horizon-2020.eu/database/data-sets) | BNCI Horizon 2020 | 45 | 7,200 |  | 58 | 256 | open | 2020 |
| [Ofner 2017 upper-limb movements](https://bnci-horizon-2020.eu/database/data-sets) | BNCI Horizon 2020 | 15 | 12,600 |  | 61 | 512 | open | 2017 |
| [High-Gamma Dataset (Schirrmeister 2017)](https://web.gin.g-node.org/robintibor/high-gamma-dataset/) | G-Node GIN | 14 | 13,484 |  | 128 | 500 | open | 2017 |
| [WAY-EEG-GAL (Kaggle Grasp-and-Lift)](https://doi.org/10.6084/m9.figshare.c.988376.v2) | Figshare | 12 | 3,936 |  | 32 | 500 | open | 2014 |

### ERP / P300

In [ ]:
#| echo: false
#| column: page
paradigm_table('ERP / P300')

| Dataset | Host | Subjects | Trials | Hours | Channels | Hz | Access | Year |
|---|---|---:|---:|---:|---:|---:|---|---|
| [bigP3BCI (Mainsah 2025)](https://physionet.org/content/bigp3bci/1.0.0/) | PhysioNet | 326 |  |  |  | 256 | open | 2025 |
| [Brain Invaders 2014a](https://doi.org/10.5281/zenodo.3266223) | Zenodo | 64 |  |  | 16 | 512 | open | 2019 |
| [Tsinghua RSVP benchmark (Zhang 2020)](http://bci.med.tsinghua.edu.cn/download.html) | Lab website | 64 | 10,240 |  | 64 | 250 | open | 2020 |
| [Won 2022 RSVP and P300 speller](https://doi.org/10.6084/m9.figshare.c.5769449.v1) | Figshare | 55 |  |  | 32 | 512 | open | 2022 |
| [OpenBMI (Lee 2019) ERP](https://doi.org/10.5524/100542) | GigaDB | 54 |  |  | 62 | 1,000 | open | 2019 |
| [Brain Invaders 2015b (multi-user)](https://doi.org/10.5281/zenodo.3267307) | Zenodo | 44 |  |  | 32 | 512 | open | 2019 |
| [Brain Invaders 2015a](https://doi.org/10.5281/zenodo.3266930) | Zenodo | 43 |  |  | 32 | 512 | open | 2019 |
| [BCI Challenge @ NER 2015 (Kaggle)](https://www.kaggle.com/competitions/inria-bci-challenge) | Kaggle | 26 | 8,840 |  | 56 | 200 | registration | 2014 |
| [Brain Invaders 2013a](https://doi.org/10.5281/zenodo.1494163) | Zenodo | 24 |  |  | 16 | 512 | open | 2019 |
| [Mobile BCI ERP (Lee 2021)](https://osf.io/r7s9b/) | OSF | 24 |  |  |  | 500 | open | 2021 |
| [AMUSE auditory spatial ERP](https://bnci-horizon-2020.eu/database/data-sets) | BNCI Horizon 2020 | 21 |  |  | 60 | 1,000 | open | 2011 |
| [Hübner 2017 label-proportion speller](http://doi.org/10.5281/zenodo.192684) | Zenodo | 13 |  |  | 31 | 1,000 | open | 2017 |
| [Sosulski 2019 auditory oddball SOA](https://freidok.uni-freiburg.de/data/154576) | FreiDok | 13 |  |  | 31 | 1,000 | open | 2019 |
| [Arico 2014 P300 speller](https://bnci-horizon-2020.eu/database/data-sets) | BNCI Horizon 2020 | 10 |  |  | 16 | 256 | open | 2014 |
| [EPFL P300 (Hoffmann 2008)](http://bci.epfl.ch/p300) | Lab website | 8 |  |  | 32 | 2,048 | open | 2008 |
| [Riccio 2013 ALS P300 speller](https://bnci-horizon-2020.eu/database/data-sets) | BNCI Horizon 2020 | 8 |  |  | 8 | 256 | open | 2013 |
| [BCI Competition III II (Wadsworth P300)](https://www.bbci.de/competition/iii/) | Lab website | 2 | 370 |  | 64 | 240 | open | 2006 |
| [BCI Competition II IIb (Wadsworth P300)](https://www.bbci.de/competition/ii/) | Lab website | 1 |  |  | 64 | 240 | open | 2004 |

### SSVEP

In [ ]:
#| echo: false
#| column: page
paradigm_table('SSVEP')

| Dataset | Host | Subjects | Trials | Hours | Channels | Hz | Access | Year |
|---|---|---:|---:|---:|---:|---:|---|---|
| [Wearable SSVEP dataset (Zhu 2021)](https://doi.org/10.6084/m9.figshare.13560281) | Figshare | 102 | 24,480 |  | 8 | 250 | open | 2021 |
| [eldBETA SSVEP (elderly)](https://doi.org/10.6084/m9.figshare.18032669) | Figshare | 100 | 6,300 |  | 64 | 1,000 | open | 2022 |
| [BETA SSVEP benchmark](http://bci.med.tsinghua.edu.cn/download.html) | Lab website | 70 | 11,200 |  | 64 | 250 | open | 2020 |
| [Dong 2023 adolescent SSVEP](https://zenodo.org/records/18847318) | Zenodo | 59 | 9,440 |  | 8 | 250 | open | 2023 |
| [OpenBMI (Lee 2019) SSVEP](https://doi.org/10.5524/100542) | GigaDB | 54 | 21,600 |  | 62 | 1,000 | open | 2019 |
| [Kim 2025 beta-range 40-class SSVEP](https://doi.org/10.6084/m9.figshare.28806815.v2) | Figshare | 40 | 9,600 |  | 31 | 1,024 | open | 2025 |
| [Tsinghua SSVEP benchmark (Wang 2016)](http://bci.med.tsinghua.edu.cn/download.html) | Lab website | 35 | 8,400 |  | 64 | 250 | open | 2016 |
| [MAMEM SSVEP dataset 1](https://dx.doi.org/10.6084/m9.figshare.2068677.v1) | Figshare | 11 |  |  | 256 | 250 | open | 2016 |
| [Nakanishi 2015 12-class SSVEP](https://github.com/mnakanishi/12JFPM_SSVEP) | GitHub | 10 | 1,800 |  | 8 | 256 | open | 2015 |

### c-VEP

In [ ]:
#| echo: false
#| column: page
paradigm_table('c-VEP')

| Dataset | Host | Subjects | Trials | Hours | Channels | Hz | Access | Year |
|---|---|---:|---:|---:|---:|---:|---|---|
| [Zheng 2024 narrow-band c-VEP](https://doi.org/10.6084/m9.figshare.24864243) | Figshare | 100 | 36,000 |  | 59 | 250 | open | 2024 |
| [Thielen 2021 zero-training c-VEP](https://doi.org/10.34973/9txv-z787) | Radboud Data Repository | 30 | 3,000 |  | 8 | 512 | open | 2021 |
| [Martínez-Cagigal 2023 p-ary c-VEP](https://doi.org/10.71569/025s-eq10) | Universidad de Valladolid repository | 16 |  |  | 16 | 256 | open | 2023 |
| [Castillos 2023 burst c-VEP](https://zenodo.org/record/8255618) | Zenodo | 12 | 2,880 |  | 32 | 500 | open | 2023 |
| [Thielen 2015 broad-band VEP](https://doi.org/10.34973/1ecz-1232) | Radboud Data Repository | 12 |  |  | 64 | 2,048 | open | 2015 |

## Clinical, sleep and resting state

### Epilepsy & clinical

In [ ]:
#| echo: false
#| column: page
paradigm_table('Epilepsy & clinical')

| Dataset | Host | Subjects | Trials | Hours | Channels | Hz | Access | Year |
|---|---|---:|---:|---:|---:|---:|---|---|
| [NMT Scalp EEG Dataset](https://dll.seecs.nust.edu.pk/downloads/) | Lab website | 2,417 |  | 625 | 19 | 200 | open | 2021 |
| [TUH Abnormal EEG Corpus (TUAB)](https://isip.piconepress.com/projects/nedc/html/tuh_eeg/) | TUH EEG | 2,329 |  | 1,142 |  |  | DUA | 2017 |
| [HMS Harmful Brain Activity Classification](https://www.kaggle.com/competitions/hms-harmful-brain-activity-classification) | Kaggle | 1,950 | 106,800 |  |  | 200 | registration | 2024 |
| [TDBRAIN](https://www.brainclinics.com/resources/tdbrain-dataset) | Lab website | 1,274 |  |  | 26 | 500 | DUA | 2022 |
| [CAUEEG](https://github.com/ipis-mjkim/caueeg-dataset) | Lab website | 1,155 |  |  | 19 | 200 | request | 2023 |
| [TUH EEG Seizure Corpus (TUSZ)](https://isip.piconepress.com/projects/nedc/html/tuh_eeg/) | TUH EEG | 675 |  | 1,475 |  |  | DUA | 2018 |
| [I-CARE (post-cardiac-arrest coma EEG)](https://physionet.org/content/i-care/2.1/) | PhysioNet | 607 |  | 32,712 |  |  | open | 2023 |
| [TUH EEG Artifact Corpus (TUAR)](https://isip.piconepress.com/projects/nedc/html/tuh_eeg/) | TUH EEG | 213 |  | 100 |  |  | DUA | 2021 |
| [TUH EEG Epilepsy Corpus (TUEP)](https://isip.piconepress.com/projects/nedc/html/tuh_eeg/) | TUH EEG | 200 |  | 627 |  |  | DUA | 2017 |
| [Iowa Parkinson's eyes-open rest (ds004584)](https://openneuro.org/datasets/ds004584) | OpenNeuro | 149 |  |  | 63 | 500 | open | 2023 |
| [SeizeIT2 wearable epilepsy dataset](https://openneuro.org/datasets/ds005873) | OpenNeuro | 125 |  | 11,640 | 2 | 250 | open | 2025 |
| [Depression rest (ds003478)](https://openneuro.org/datasets/ds003478) | OpenNeuro | 122 |  |  | 64 | 500 | open | 2021 |
| [EEG data for ADHD/control children](https://ieee-dataport.org/open-access/eeg-data-adhd-control-children) | IEEE DataPort | 121 |  |  | 19 | 128 | registration | 2020 |
| [Alzheimer's, FTD and healthy resting EEG (ds004504)](https://openneuro.org/datasets/ds004504) | OpenNeuro | 88 |  | 19 | 19 | 500 | open | 2023 |
| [First-episode psychosis rest (ds003944)](https://openneuro.org/datasets/ds003944) | OpenNeuro | 82 |  |  | 61 | 1,000 | open | 2022 |
| [Helsinki neonatal EEG seizure dataset](https://zenodo.org/records/2547147) | Zenodo | 79 |  |  | 19 | 256 | open | 2019 |
| [MDD patients and healthy controls EEG (Mumtaz)](https://figshare.com/articles/dataset/EEG_Data_New/4244171) | Figshare | 64 |  |  | 19 | 256 | open | 2016 |
| [MODMA (multi-modal mental-disorder dataset)](http://modma.lzu.edu.cn/data/index/) | Lab website | 53 |  |  | 128 | 250 | DUA | 2022 |
| [Parkinson's 3-stim oddball and rest (ds003490)](https://openneuro.org/datasets/ds003490) | OpenNeuro | 50 |  |  | 64 | 500 | open | 2021 |
| [TUH EEG Slowing Corpus (TUSL)](https://isip.piconepress.com/projects/nedc/html/tuh_eeg/) | TUH EEG | 38 |  |  |  |  | DUA | 2017 |
| [UC San Diego Parkinson's resting EEG (ds002778)](https://openneuro.org/datasets/ds002778) | OpenNeuro | 31 |  |  | 40 | 512 | open | 2020 |
| [EEG in schizophrenia (RepOD)](https://doi.org/10.18150/repod.0107441) | RepOD | 28 |  | 7 | 19 | 250 | open | 2017 |
| [CHB-MIT Scalp EEG Database](https://physionet.org/content/chbmit/1.0.0/) | PhysioNet | 22 |  | 983 | 23 | 256 | open | 2010 |
| [Siena Scalp EEG Database](https://physionet.org/content/siena-scalp-eeg/1.0.0/) | PhysioNet | 14 |  | 128 | 29 | 512 | open | 2020 |
| [Bonn University EEG (sets A-E)](https://www.ukbonn.de/epileptologie/arbeitsgruppen/ag-lehnertz-neurophysik/downloads/) | Lab website |  | 500 | 3.3 | 1 | 174 | open | 2001 |
| [TUH EEG Events Corpus (TUEV)](https://isip.piconepress.com/projects/nedc/html/tuh_eeg/) | TUH EEG |  |  |  |  |  | DUA | 2015 |

### Sleep

In [ ]:
#| echo: false
#| column: page
paradigm_table('Sleep')

| Dataset | Host | Subjects | Trials | Hours | Channels | Hz | Access | Year |
|---|---|---:|---:|---:|---:|---:|---|---|
| [Human Sleep Project (HSP)](https://bdsp.io/content/hsp/) | BDSP | 90,166 |  |  | 6 |  | DUA | 2026 |
| [Sleep Heart Health Study (SHHS)](https://sleepdata.org/datasets/shhs) | NSRR | 5,804 |  |  | 2 | 125 | DUA | 1997 |
| [NCH Sleep DataBank (NCHSDB)](https://sleepdata.org/datasets/nchsdb) | NSRR | 3,673 |  |  |  |  | DUA | 2022 |
| [Osteoporotic Fractures in Men sleep (MrOS)](https://sleepdata.org/datasets/mros) | NSRR | 2,911 |  |  | 4 | 256 | DUA | 2011 |
| [Multi-Ethnic Study of Atherosclerosis sleep (MESA)](https://sleepdata.org/datasets/mesa) | NSRR | 2,056 |  |  | 3 | 256 | DUA | 2015 |
| [PhysioNet/CinC Challenge 2018 (You Snooze You Win)](https://physionet.org/content/challenge-2018/1.0.0/) | PhysioNet | 1,985 |  | 15,261 | 6 | 200 | open | 2018 |
| [Cleveland Family Study (CFS)](https://sleepdata.org/datasets/cfs) | NSRR | 735 |  |  | 4 | 128 | DUA | 1995 |
| [Childhood Adenotonsillectomy Trial (CHAT)](https://sleepdata.org/datasets/chat) | NSRR | 464 |  |  | 10 | 200 | DUA | 2013 |
| [Montreal Archive of Sleep Studies (MASS)](http://ceams-carsm.ca/mass/) | Borealis | 200 |  |  |  |  | request | 2014 |
| [Haaglanden Medisch Centrum sleep staging (HMC)](https://physionet.org/content/hmc-sleep-staging/1.1/) | PhysioNet | 151 |  | 1,144 | 4 | 256 | open | 2022 |
| [ISRUC-Sleep](https://sleeptight.isr.uc.pt/) | Lab website | 118 |  |  | 6 | 200 | open | 2016 |
| [CAP Sleep Database](https://physionet.org/content/capslpdb/1.0.0/) | PhysioNet | 108 |  |  |  |  | open | 2012 |
| [Bitbrain Open Access Sleep (BOAS)](https://openneuro.org/datasets/ds005555) | OpenNeuro | 100 |  |  | 6 | 256 | open | 2024 |
| [Sleep-EDF Expanded](https://physionet.org/content/sleep-edfx/1.0.0/) | PhysioNet | 100 |  | 3,849 | 2 | 100 | open | 2013 |
| [Dreem Open Dataset - Obstructive (DOD-O)](https://zenodo.org/records/15900394) | Zenodo | 55 |  |  | 8 | 250 | open | 2020 |
| [Dreem Open Dataset - Healthy (DOD-H)](https://zenodo.org/records/15900394) | Zenodo | 25 |  |  | 12 | 250 | open | 2020 |
| [Wisconsin Sleep Cohort (WSC)](https://sleepdata.org/datasets/wsc) | NSRR |  |  |  |  |  | DUA | 2009 |

### Resting state

In [ ]:
#| echo: false
#| column: page
paradigm_table('Resting state')

| Dataset | Host | Subjects | Trials | Hours | Channels | Hz | Access | Year |
|---|---|---:|---:|---:|---:|---:|---|---|
| [Dortmund Vital Study resting EEG (ds005385)](https://openneuro.org/datasets/ds005385) | OpenNeuro | 608 |  |  | 64 | 1,000 | open | 2024 |
| [Cuban Human Brain Mapping Project (CHBMP)](https://chbmp-open.loris.ca) | Synapse | 250 |  |  |  | 200 | registration | 2021 |
| [MPI Leipzig Mind-Brain-Body (LEMON)](https://fcon_1000.projects.nitrc.org/indi/retro/MPI_LEMON.html) | INDI (NITRC) | 216 |  | 58 | 62 | 2,500 | open | 2019 |
| [SRM resting-state EEG (ds003775)](https://openneuro.org/datasets/ds003775) | OpenNeuro | 111 |  |  | 64 | 1,024 | open | 2022 |
| [Test-retest resting and cognitive state EEG (ds004148)](https://openneuro.org/datasets/ds004148) | OpenNeuro | 60 |  |  | 64 | 500 | open | 2022 |

### Large corpus

In [ ]:
#| echo: false
#| column: page
paradigm_table('Large corpus')

| Dataset | Host | Subjects | Trials | Hours | Channels | Hz | Access | Year |
|---|---|---:|---:|---:|---:|---:|---|---|
| [Harvard Electroencephalography Database (HEEDB)](https://bdsp.io/content/harvard-eeg-db/) | BDSP | 109,178 |  |  |  |  | DUA | 2025 |
| [TUH EEG Corpus (TUEG)](https://isip.piconepress.com/projects/nedc/html/tuh_eeg/) | TUH EEG | 14,987 |  | 27,074 |  |  | DUA | 2016 |
| [Healthy Brain Network EEG (HBN-EEG)](https://openneuro.org/datasets/ds005505) | OpenNeuro | 3,155 |  |  | 129 | 500 | open | 2024 |

## Cognition, emotion and perception

### Affective

In [ ]:
#| echo: false
#| column: page
paradigm_table('Affective')

| Dataset | Host | Subjects | Trials | Hours | Channels | Hz | Access | Year |
|---|---|---:|---:|---:|---:|---:|---|---|
| [FACED](https://doi.org/10.7303/syn50614194) | Synapse | 123 | 3,444 |  | 32 |  | registration | 2023 |
| [EAV](https://zenodo.org/records/10205702) | Zenodo | 42 | 8,400 | 47 | 30 | 500 | DUA | 2024 |
| [AMIGOS](http://www.eecs.qmul.ac.uk/mmv/datasets/amigos/) | Lab website | 40 | 788 |  | 14 | 128 | DUA | 2018 |
| [SEED-SD](https://bcmi.sjtu.edu.cn/home/seed/seed-SD.html) | Lab website | 40 | 2,880 |  | 62 | 1,000 | DUA | 2025 |
| [Imagined Emotion (ds003004)](https://openneuro.org/datasets/ds003004) | OpenNeuro | 34 |  |  | 224 | 256 | open | 2009 |
| [DEAP](https://www.eecs.qmul.ac.uk/mmv/datasets/deap/) | Lab website | 32 | 1,280 | 21 | 32 | 512 | DUA | 2012 |
| [MAHNOB-HCI](https://mahnob-db.eu/hci-tagging/) | Lab website | 27 | 540 |  | 32 |  | registration | 2012 |
| [DREAMER](https://zenodo.org/records/546113) | Zenodo | 23 | 414 |  | 14 | 128 | request | 2018 |
| [SEED-VII](https://bcmi.sjtu.edu.cn/home/seed/seed-vii.html) | Lab website | 20 | 1,600 | 78 | 62 | 1,000 | DUA | 2025 |
| [SEED-V](https://bcmi.sjtu.edu.cn/home/seed/seed-v.html) | Lab website | 16 | 720 |  | 62 |  | DUA | 2021 |
| [SEED](https://bcmi.sjtu.edu.cn/home/seed/seed.html) | Lab website | 15 | 675 | 45 | 62 | 1,000 | DUA | 2015 |
| [SEED-IV](https://bcmi.sjtu.edu.cn/home/seed/seed-iv.html) | Lab website | 15 | 1,080 |  | 62 |  | DUA | 2019 |
| [SEED-FRA](https://bcmi.sjtu.edu.cn/home/seed/seed-FRA.html) | Lab website | 8 | 504 |  | 62 | 1,000 | DUA | 2022 |
| [SEED-GER](https://bcmi.sjtu.edu.cn/home/seed/seed-GER.html) | Lab website | 8 |  |  | 62 | 1,000 | DUA | 2022 |

### Visual perception

In [ ]:
#| echo: false
#| column: page
paradigm_table('Visual perception')

| Dataset | Host | Subjects | Trials | Hours | Channels | Hz | Access | Year |
|---|---|---:|---:|---:|---:|---:|---|---|
| [THINGS-EEG1](https://openneuro.org/datasets/ds003825) | OpenNeuro | 50 |  | 50 | 63 | 1,000 | open | 2022 |
| [Alljoined-1.6M](https://huggingface.co/datasets/Alljoined/Alljoined-1.6M) | Hugging Face | 20 | 1,670,400 | 160 | 32 | 256 | open | 2025 |
| [SEED-DV](https://bcmi.sjtu.edu.cn/home/eeg2video/) | Lab website | 20 |  |  | 62 | 200 | DUA | 2024 |
| [Wakeman-Henson faces (ds002718)](https://openneuro.org/datasets/ds002718) | OpenNeuro | 18 |  |  | 70 | 250 | open | 2015 |
| [EEG-ImageNet](https://github.com/Promise-Z5Q2SQ/EEG-ImageNet-Dataset) | Tsinghua Cloud | 16 | 87,850 |  | 62 | 1,000 | open | 2024 |
| [Go/no-go categorization (ds002680)](https://openneuro.org/datasets/ds002680) | OpenNeuro | 14 |  |  | 31 | 1,000 | open | 2004 |
| [Object Category EEG Dataset (OCED)](https://purl.stanford.edu/tc919dd5388) | Stanford Digital Repository | 10 | 51,840 |  | 128 | 1,000 | open | 2015 |
| [THINGS-EEG2](https://osf.io/3jk45/) | OSF | 10 | 821,600 |  | 64 | 1,000 | open | 2022 |
| [Alljoined1](https://huggingface.co/datasets/Alljoined/05_125) | Hugging Face | 8 | 46,080 |  | 64 | 512 | open | 2024 |
| [Spampinato EEG-ImageNet (PeRCeiVe)](https://github.com/perceivelab/eeg_visual_classification) | Google Drive | 6 | 11,964 |  | 128 | 1,000 | open | 2017 |

### Auditory & speech

In [ ]:
#| echo: false
#| column: page
paradigm_table('Auditory & speech')

| Dataset | Host | Subjects | Trials | Hours | Channels | Hz | Access | Year |
|---|---|---:|---:|---:|---:|---:|---|---|
| [SparrKULee](https://doi.org/10.48804/K3VSND) | KU Leuven RDR | 85 |  | 168 | 64 | 8,192 | open | 2024 |
| [DTU hearing-impaired AAD](https://zenodo.org/records/3618205) | Zenodo | 44 |  |  |  |  | open | 2020 |
| [FEIS](https://zenodo.org/records/3554128) | Zenodo | 21 |  |  | 14 |  | open | 2019 |
| [MUSIN-G (ds003774)](https://openneuro.org/datasets/ds003774) | OpenNeuro | 20 | 240 |  | 129 | 250 | open | 2021 |
| [NMED-T](https://purl.stanford.edu/jn859kj8079) | Stanford Digital Repository | 20 | 200 |  | 128 | 1,000 | open | 2017 |
| [Natural speech (Broderick/Di Liberto, ds004408)](https://openneuro.org/datasets/ds004408) | OpenNeuro | 19 | 380 |  | 128 | 512 | open | 2018 |
| [DTU AAD](https://zenodo.org/records/1199011) | Zenodo | 18 |  |  | 64 | 512 | open | 2017 |
| [KU Leuven AAD](https://zenodo.org/records/4004271) | Zenodo | 16 | 320 | 19 | 64 | 8,192 | open | 2016 |
| [BCI Competition 2020 Track 3 imagined speech](https://osf.io/pq7vb/) | OSF | 15 | 5,250 |  | 64 |  | open | 2022 |
| [Kara One](https://www.cs.toronto.edu/~complingweb/data/karaOne/karaOne.html) | Lab website | 14 |  |  | 62 |  | open | 2015 |
| [AV-GC-AAD](https://zenodo.org/records/11058711) | Zenodo | 13 | 104 | 17 | 64 | 8,192 | open | 2024 |
| [Inner Speech (Nieto)](https://openneuro.org/datasets/ds003626) | OpenNeuro | 10 | 5,640 | 9 | 128 | 1,024 | open | 2022 |
| [OpenMIIR](https://github.com/sstober/openmiir) | Lab website | 10 |  |  | 64 | 512 | open | 2015 |
| [Chisco](https://openneuro.org/datasets/ds005170) | OpenNeuro | 5 |  |  | 128 | 1,000 | open | 2024 |

### Language

In [ ]:
#| echo: false
#| column: page
paradigm_table('Language')

| Dataset | Host | Subjects | Trials | Hours | Channels | Hz | Access | Year |
|---|---|---:|---:|---:|---:|---:|---|---|
| [Alice EEG (Brennan)](https://deepblue.lib.umich.edu/data/concern/data_sets/bg257f92t) | Deep Blue Data | 49 |  |  | 61 | 500 | open | 2019 |
| [TMNRED](https://openneuro.org/datasets/ds005383) | OpenNeuro | 30 |  | 10 | 30 | 200 | open | 2025 |
| [DERCo](https://osf.io/rkqbu/) | OSF | 22 |  |  | 32 | 1,000 | open | 2024 |
| [ZuCo 2.0](https://osf.io/2urht/) | OSF | 18 |  |  | 128 | 500 | open | 2020 |
| [ZuCo 1.0](https://osf.io/q3zws/) | OSF | 12 |  |  | 128 | 500 | open | 2018 |
| [ChineseEEG](https://openneuro.org/datasets/ds004952) | OpenNeuro | 10 |  | 130 | 128 | 1,000 | open | 2024 |

### Cognitive

In [ ]:
#| echo: false
#| column: page
paradigm_table('Cognitive')

| Dataset | Host | Subjects | Trials | Hours | Channels | Hz | Access | Year |
|---|---|---:|---:|---:|---:|---:|---|---|
| [STEW](https://ieee-dataport.org/open-access/stew-simultaneous-task-eeg-workload-dataset) | IEEE DataPort | 48 |  | 4 | 14 | 128 | registration | 2018 |
| [ERP CORE](https://osf.io/thsqg/) | OSF | 40 |  |  | 30 | 1,024 | open | 2021 |
| [EEGMAT (mental arithmetic)](https://physionet.org/content/eegmat/1.0.0/) | PhysioNet | 36 |  |  |  |  | open | 2019 |
| [COG-BCI](https://zenodo.org/records/7413650) | Zenodo | 29 |  | 100 | 64 | 500 | open | 2023 |
| [Sustained-attention driving (Cao)](https://doi.org/10.6084/m9.figshare.6427334.v5) | Figshare | 27 |  | 93 | 32 | 500 | open | 2019 |
| [Shin 2018 n-back/DSR/WG (EEG-NIRS)](https://doi.org/10.14279/depositonce-5830.2) | DepositOnce (TU Berlin) | 26 |  |  | 30 | 1,000 | open | 2018 |
| [SEED-VIG](https://bcmi.sjtu.edu.cn/home/seed/seed-vig.html) | Lab website | 23 |  | 46 | 18 | 1,000 | DUA | 2017 |

## Corrections and additions

Spotted a wrong number or a missing dataset? Edit the CSV and open a pull request, or [open an issue](https://github.com/bkowshik/temple-of-eeg-decoding/issues) with a link to the dataset's page.